In [0]:
from pyspark.sql.functions import * 

#fetching last_run_time from last successful run from control table to process only new data

last_run_time = spark.sql("select last_run_time from data_engineering.table_metadata.Control_Table where pipeline_name='medallion_architecture_sales_pipeline'").collect()[0][0]

print(f"last_run_time:{last_run_time}")

#only new data is taken from bronze table

df = spark.table("data_engineering.bronze_layer.bronze_sales").filter(col("ingestion_time") > last_run_time)

df = df.withColumn("sales", expr("try_cast(sales as double)"))

df= df.withColumn("sales", coalesce(col("sales"), lit(0)))

df = df.fillna({"city": "unknown"})

df = df.withColumn("city", lower(col("city")))

print(f"record count before dropduplicate:{df.count()}")

#Removing duplicates
df = df.dropDuplicates(["id", "name", "age", "city", "sales"])

print(f"record count after dropduplicate:{df.count()}")

df.write.format("delta").mode("append").saveAsTable("data_engineering.silver_layer.silver_sales")